# SmartSpend Model Training Notebook

This notebook trains and evaluates the two machine learning components described in the SmartSpend proposal:

1. **Transaction categorisation model:** TF-IDF vectorisation + Logistic Regression for classifying MTN Mobile Money expense descriptions into the 10 fixed expense categories.
2. **Financial prediction models:** XGBoost regression models for predicting month-end expense and month-end income, from which the overspend risk score can be derived.

The notebook also includes extensive data visualisation, feature engineering, model evaluation, confusion matrices, correlation analysis, and reusable retraining functions that can later be moved into the FastAPI backend.

In [ ]:
# Core libraries
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.inspection import permutation_importance
from pandas.plotting import scatter_matrix

try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None
    print("xgboost is not installed. Install it with: pip install xgboost")

# Paths
SMS_DATASET = "smartspend_initial_synthetic_momo_sms_dataset.csv"
PRED_DATASET = "smartspend_initial_synthetic_prediction_demo_dataset.csv"
MODEL_DIR = "smartspend_initial_models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Notebook display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## 1. Load datasets

The categorisation dataset is proposal-aligned: it is **synthetic**, **Rwanda-contextualised**, and **MTN MoMo-only** for the current version.

The prediction dataset is a **synthetic demo/scaffold** only. It exists so the XGBoost training and backend integration pipeline can be prepared. In the real application, this prediction model should be retrained using accumulated per-user income and expense history, as stated in the proposal.

In [ ]:
sms_df = pd.read_csv(SMS_DATASET)
pred_df = pd.read_csv(PRED_DATASET)

print("SMS dataset shape:", sms_df.shape)
print("Prediction dataset shape:", pred_df.shape)

display(sms_df.head())
display(pred_df.head())

print("SMS columns:", list(sms_df.columns))
print("Prediction columns:", list(pred_df.columns))

## 2. Data quality checks

These checks confirm that the training data is usable before model training.

In [ ]:
print("Missing values in SMS dataset:")
display(sms_df.isna().sum().to_frame("missing_count"))

print("Missing values in prediction dataset:")
display(pred_df.isna().sum().to_frame("missing_count"))

print("Duplicate SMS rows:", sms_df.duplicated().sum())
print("Duplicate prediction rows:", pred_df.duplicated().sum())

if "sender" in sms_df.columns:
    print("Unique SMS senders:", sms_df["sender"].unique())
if "provider" in sms_df.columns:
    print("Unique providers:", sms_df["provider"].unique())

## 3. Data engineering

For categorisation, only **SENT** transactions are used because the 10 fixed category labels apply to expenses. Received transactions belong to the income side of the dashboard and prediction model, not the expense categorisation model.

In [ ]:
expense_df = sms_df[sms_df["transaction_type"] == "SENT"].copy()

# The text used by TF-IDF combines the extracted description and raw SMS message.
expense_df["description"] = expense_df["description"].fillna("")
expense_df["raw_sms"] = expense_df["raw_sms"].fillna("")
expense_df["text"] = (expense_df["description"] + " " + expense_df["raw_sms"]).str.strip()
expense_df["text_length"] = expense_df["text"].str.len()
expense_df["word_count"] = expense_df["text"].str.split().str.len()

# Dataset column naming safeguard: proposal says amount in RWF; the generated dataset uses amount_rwf.
amount_col = "amount_rwf" if "amount_rwf" in expense_df.columns else "amount"

print("Expense training rows:", len(expense_df))
display(expense_df[["raw_sms", "description", amount_col, "category", "text_length", "word_count"]].head())

## 4. Data visualisations for categorisation dataset

In [ ]:
# 4.1 Transaction type distribution
transaction_counts = sms_df["transaction_type"].value_counts()
plt.figure(figsize=(7, 5))
transaction_counts.plot(kind="bar")
plt.title("Transaction Type Distribution")
plt.xlabel("Transaction Type")
plt.ylabel("Number of Records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 4.2 Expense category distribution
category_counts = expense_df["category"].value_counts().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
category_counts.plot(kind="bar")
plt.title("Expense Category Distribution - Synthetic MTN MoMo Dataset")
plt.xlabel("Expense Category")
plt.ylabel("Number of Records")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 4.3 Category percentages
category_percentages = (category_counts / category_counts.sum() * 100).round(2)
display(category_percentages.to_frame("percentage_of_expense_dataset"))

In [ ]:
# 4.4 Amount distribution
plt.figure(figsize=(9, 5))
expense_df[amount_col].plot(kind="hist", bins=30)
plt.title("Expense Amount Distribution")
plt.xlabel("Amount (RWF)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# 4.5 Amount distribution by category
plt.figure(figsize=(12, 6))
expense_df.boxplot(column=amount_col, by="category", rot=45)
plt.title("Expense Amount Distribution by Category")
plt.suptitle("")
plt.xlabel("Expense Category")
plt.ylabel("Amount (RWF)")
plt.tight_layout()
plt.show()

# 4.6 Average amount by category
mean_amount_by_category = expense_df.groupby("category")[amount_col].mean().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
mean_amount_by_category.plot(kind="bar")
plt.title("Average Expense Amount by Category")
plt.xlabel("Expense Category")
plt.ylabel("Average Amount (RWF)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 4.7 Text length distribution
plt.figure(figsize=(9, 5))
expense_df["text_length"].plot(kind="hist", bins=30)
plt.title("SMS Text Length Distribution")
plt.xlabel("Text Length")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# 4.8 Word count distribution by category
plt.figure(figsize=(12, 6))
expense_df.boxplot(column="word_count", by="category", rot=45)
plt.title("Word Count Distribution by Category")
plt.suptitle("")
plt.xlabel("Expense Category")
plt.ylabel("Word Count")
plt.tight_layout()
plt.show()

# 4.9 Average text length by category
avg_text_length = expense_df.groupby("category")["text_length"].mean().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
avg_text_length.plot(kind="bar")
plt.title("Average SMS Text Length by Category")
plt.xlabel("Expense Category")
plt.ylabel("Average Text Length")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Model architecture

### 5.1 Categorisation model architecture

The categorisation model is not a neural network, so it has no layers or activation functions. This is intentional and consistent with the proposal. The architecture is:

1. Input: short MTN MoMo SMS text and extracted description.
2. Feature extraction: TF-IDF vectorisation with unigrams and bigrams.
3. Classifier: Logistic Regression.
4. Output: one of the 10 fixed expense categories, plus class probabilities used as confidence scores.

Optimisation technique: Logistic Regression uses a numerical optimiser internally to minimise regularised multiclass cross-entropy loss.

### 5.2 Prediction model architecture

The prediction model is also not a neural network. It uses XGBoost regressors, which are gradient-boosted decision tree ensembles. The architecture is:

1. Input: engineered monthly financial features.
2. Model: XGBoost regression trees.
3. Outputs: predicted month-end expense and predicted month-end income.
4. Derived output: overspend risk score based on predicted expense compared with predicted income.

There are no activation functions because XGBoost is tree-based, not layer-based.

## 6. Train and evaluate the categorisation model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    expense_df["text"],
    expense_df["category"],
    test_size=0.2,
    random_state=42,
    stratify=expense_df["category"]
)

category_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

category_model.fit(X_train, y_train)
y_pred = category_model.predict(X_test)

cat_metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision_macro": float(precision_score(y_test, y_pred, average="macro", zero_division=0)),
    "recall_macro": float(recall_score(y_test, y_pred, average="macro", zero_division=0)),
    "f1_macro": float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
    "precision_weighted": float(precision_score(y_test, y_pred, average="weighted", zero_division=0)),
    "recall_weighted": float(recall_score(y_test, y_pred, average="weighted", zero_division=0)),
    "f1_weighted": float(f1_score(y_test, y_pred, average="weighted", zero_division=0))
}

print(json.dumps(cat_metrics, indent=2))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# 6.1 Confusion matrix
labels = sorted(expense_df["category"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, xticks_rotation=45, values_format="d")
plt.title("Confusion Matrix - Expense Categorisation Model")
plt.tight_layout()
plt.show()

# 6.2 Normalised confusion matrix
cm_norm = confusion_matrix(y_test, y_pred, labels=labels, normalize="true")
fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=labels)
disp.plot(ax=ax, xticks_rotation=45, values_format=".2f")
plt.title("Normalised Confusion Matrix - Expense Categorisation Model")
plt.tight_layout()
plt.show()

In [ ]:
# 6.3 Per-category F1 score visualisation
report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
per_class_rows = []
for label in labels:
    row = report_dict[label].copy()
    row["category"] = label
    per_class_rows.append(row)
per_class_df = pd.DataFrame(per_class_rows).set_index("category")
display(per_class_df)

plt.figure(figsize=(12, 6))
per_class_df["f1-score"].sort_values().plot(kind="bar")
plt.title("F1 Score by Expense Category")
plt.xlabel("Expense Category")
plt.ylabel("F1 Score")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# 6.4 Model confidence distribution
if hasattr(category_model.named_steps["classifier"], "predict_proba"):
    proba = category_model.predict_proba(X_test)
    confidence = proba.max(axis=1)
    confidence_df = pd.DataFrame({
        "true_category": y_test.values,
        "predicted_category": y_pred,
        "confidence": confidence,
        "correct": y_test.values == y_pred
    })
    display(confidence_df.head())

    plt.figure(figsize=(9, 5))
    confidence_df["confidence"].plot(kind="hist", bins=30)
    plt.title("Prediction Confidence Distribution")
    plt.xlabel("Maximum Predicted Probability")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 5))
    confidence_df.boxplot(column="confidence", by="correct")
    plt.title("Prediction Confidence by Correctness")
    plt.suptitle("")
    plt.xlabel("Prediction Correct")
    plt.ylabel("Confidence")
    plt.tight_layout()
    plt.show()

In [ ]:
# 6.5 Top TF-IDF terms per category from Logistic Regression coefficients
tfidf = category_model.named_steps["tfidf"]
clf = category_model.named_steps["classifier"]
feature_names = np.array(tfidf.get_feature_names_out())

rows = []
for class_index, class_label in enumerate(clf.classes_):
    top_indices = np.argsort(clf.coef_[class_index])[-10:][::-1]
    rows.append({
        "category": class_label,
        "top_terms": ", ".join(feature_names[top_indices])
    })

top_terms_df = pd.DataFrame(rows)
display(top_terms_df)

## 7. Data visualisations for prediction dataset

These charts are for the synthetic prediction scaffold. They are useful for checking the planned API/model pipeline, but real evaluation should use accumulated user transaction history during the pilot.

In [ ]:
numeric_pred_cols = pred_df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric prediction columns:", numeric_pred_cols)

display(pred_df[numeric_pred_cols].describe().T)

# 7.1 Distribution of target month-end expense
plt.figure(figsize=(9, 5))
pred_df["target_month_end_expense"].plot(kind="hist", bins=30)
plt.title("Target Month-End Expense Distribution")
plt.xlabel("Target Month-End Expense (RWF)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# 7.2 Distribution of target month-end income
plt.figure(figsize=(9, 5))
pred_df["target_month_end_income"].plot(kind="hist", bins=30)
plt.title("Target Month-End Income Distribution")
plt.xlabel("Target Month-End Income (RWF)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# 7.3 Distribution of target overspend risk
if "target_overspend_risk" in pred_df.columns:
    plt.figure(figsize=(9, 5))
    pred_df["target_overspend_risk"].plot(kind="hist", bins=30)
    plt.title("Target Overspend Risk Distribution")
    plt.xlabel("Overspend Risk")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

In [ ]:
# 7.4 Correlation matrix for numeric columns
corr = pred_df[numeric_pred_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr.values, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticklabels(corr.columns)
plt.colorbar(im, ax=ax)
plt.title("Correlation Matrix - Prediction Dataset Numeric Features")
plt.tight_layout()
plt.show()

display(corr.round(3))

In [ ]:
# 7.5 Feature correlations with prediction targets
targets = ["target_month_end_expense", "target_month_end_income"]
if "target_overspend_risk" in pred_df.columns:
    targets.append("target_overspend_risk")

for target in targets:
    feature_corr = corr[target].drop(index=[t for t in targets if t in corr.index], errors="ignore").sort_values(key=lambda s: s.abs(), ascending=False)
    display(feature_corr.to_frame(f"correlation_with_{target}"))

    plt.figure(figsize=(10, 6))
    feature_corr.head(15).plot(kind="bar")
    plt.title(f"Top Feature Correlations with {target}")
    plt.xlabel("Feature")
    plt.ylabel("Correlation")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# 7.6 Scatter plots for important feature-target relationships
possible_features = [
    "income_received_to_date",
    "expense_to_date",
    "historical_monthly_expense_avg",
    "historical_monthly_income_avg",
    "day_of_month"
]
for feature in [f for f in possible_features if f in pred_df.columns]:
    plt.figure(figsize=(7, 5))
    plt.scatter(pred_df[feature], pred_df["target_month_end_expense"], alpha=0.6)
    plt.title(f"{feature} vs Target Month-End Expense")
    plt.xlabel(feature)
    plt.ylabel("Target Month-End Expense")
    plt.tight_layout()
    plt.show()

# 7.7 Scatter matrix for selected columns
selected_for_scatter = [c for c in possible_features + ["target_month_end_expense", "target_month_end_income"] if c in pred_df.columns]
if len(selected_for_scatter) >= 3:
    scatter_matrix(pred_df[selected_for_scatter], figsize=(14, 14), diagonal="hist")
    plt.suptitle("Scatter Matrix - Selected Prediction Features and Targets")
    plt.tight_layout()
    plt.show()

## 8. Train and evaluate prediction models

In [ ]:
if XGBRegressor is None:
    raise ImportError("xgboost is required for this section. Install with: pip install xgboost")

feature_cols = [
    c for c in pred_df.columns
    if c not in ["user_id", "month", "target_month_end_expense", "target_month_end_income", "target_overspend_risk", "is_synthetic_demo"]
]

X = pred_df[feature_cols]
y_expense = pred_df["target_month_end_expense"]
y_income = pred_df["target_month_end_income"]

X_train_p, X_test_p, y_exp_train, y_exp_test, y_inc_train, y_inc_test = train_test_split(
    X, y_expense, y_income, test_size=0.2, random_state=42
)

expense_model = XGBRegressor(
    n_estimators=40,
    max_depth=3,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    objective="reg:squarederror",
    n_jobs=1
)
income_model = XGBRegressor(
    n_estimators=40,
    max_depth=3,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    objective="reg:squarederror",
    n_jobs=1
)

expense_model.fit(X_train_p, y_exp_train)
income_model.fit(X_train_p, y_inc_train)

exp_pred = expense_model.predict(X_test_p)
inc_pred = income_model.predict(X_test_p)

def regression_metrics(y_true, y_hat):
    rmse = mean_squared_error(y_true, y_hat) ** 0.5
    return {
        "MAE": float(mean_absolute_error(y_true, y_hat)),
        "RMSE": float(rmse),
        "R2": float(r2_score(y_true, y_hat))
    }

prediction_metrics = {
    "expense_model": regression_metrics(y_exp_test, exp_pred),
    "income_model": regression_metrics(y_inc_test, inc_pred)
}
print(json.dumps(prediction_metrics, indent=2))

In [ ]:
# 8.1 Actual vs predicted plots
plt.figure(figsize=(7, 6))
plt.scatter(y_exp_test, exp_pred, alpha=0.7)
plt.title("Actual vs Predicted Month-End Expense")
plt.xlabel("Actual Expense")
plt.ylabel("Predicted Expense")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 6))
plt.scatter(y_inc_test, inc_pred, alpha=0.7)
plt.title("Actual vs Predicted Month-End Income")
plt.xlabel("Actual Income")
plt.ylabel("Predicted Income")
plt.tight_layout()
plt.show()

# 8.2 Residual distributions
expense_residuals = y_exp_test - exp_pred
income_residuals = y_inc_test - inc_pred

plt.figure(figsize=(9, 5))
pd.Series(expense_residuals).plot(kind="hist", bins=30)
plt.title("Expense Model Residual Distribution")
plt.xlabel("Actual - Predicted Expense")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
pd.Series(income_residuals).plot(kind="hist", bins=30)
plt.title("Income Model Residual Distribution")
plt.xlabel("Actual - Predicted Income")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# 8.3 XGBoost feature importance
expense_importance = pd.Series(expense_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
income_importance = pd.Series(income_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

display(expense_importance.to_frame("expense_model_importance"))
display(income_importance.to_frame("income_model_importance"))

plt.figure(figsize=(12, 6))
expense_importance.head(15).plot(kind="bar")
plt.title("Top Feature Importances - Expense Prediction Model")
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
income_importance.head(15).plot(kind="bar")
plt.title("Top Feature Importances - Income Prediction Model")
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 8.4 Overspend risk calculation from predictions
# This is a simple interpretable derivation for the prototype. It can be refined during pilot evaluation.

def calculate_overspend_risk(predicted_expense, predicted_income):
    if predicted_income <= 0:
        return 100.0
    ratio = predicted_expense / predicted_income
    return float(np.clip(ratio * 100, 0, 100))

risk_predictions = [calculate_overspend_risk(e, i) for e, i in zip(exp_pred, inc_pred)]
risk_df = pd.DataFrame({
    "actual_expense": y_exp_test.values,
    "predicted_expense": exp_pred,
    "actual_income": y_inc_test.values,
    "predicted_income": inc_pred,
    "predicted_overspend_risk": risk_predictions
})
display(risk_df.head(10))

plt.figure(figsize=(9, 5))
risk_df["predicted_overspend_risk"].plot(kind="hist", bins=20)
plt.title("Predicted Overspend Risk Distribution")
plt.xlabel("Predicted Overspend Risk (%)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 9. Save model artefacts

These files can be consumed by the FastAPI backend.

In [ ]:
joblib.dump(category_model, os.path.join(MODEL_DIR, "smartspend_category_model.joblib"))
joblib.dump(expense_model, os.path.join(MODEL_DIR, "smartspend_expense_prediction_model.joblib"))
joblib.dump(income_model, os.path.join(MODEL_DIR, "smartspend_income_prediction_model.joblib"))

all_metrics = {
    "categorisation_model": cat_metrics,
    "prediction_models": prediction_metrics,
    "note": "Prediction dataset is synthetic demo/scaffold. Real pilot data should be used for final model validation."
}

with open(os.path.join(MODEL_DIR, "initial_metrics.json"), "w") as f:
    json.dump(all_metrics, f, indent=2)

print("Saved model artefacts to:", MODEL_DIR)
print(os.listdir(MODEL_DIR))

## 10. Retraining functions for future backend API integration

These functions are deliberately written in a backend-friendly style. They can later be moved into FastAPI services and called from asynchronous retraining jobs.

Proposal alignment:
- Categorisation retraining uses the base synthetic MTN MoMo dataset plus user-confirmed corrections.
- Prediction retraining should use accumulated user income and expense history.
- The MVP should not use cross-user training unless ethical approval and consent are updated.

In [ ]:
def retrain_categorisation_model(
    base_dataset_path,
    corrections_dataset_path=None,
    output_path=os.path.join(MODEL_DIR, "smartspend_category_model_retrained.joblib")
):
    """Retrain TF-IDF + Logistic Regression using the base dataset plus optional user corrections.

    Expected correction columns: raw_sms, description, amount, transaction_type, category.
    Only SENT transactions are used for the 10 expense-category classifier.
    """
    base = pd.read_csv(base_dataset_path)
    frames = [base]

    if corrections_dataset_path is not None and os.path.exists(corrections_dataset_path):
        corrections = pd.read_csv(corrections_dataset_path)
        frames.append(corrections)

    combined = pd.concat(frames, ignore_index=True)
    combined = combined[combined["transaction_type"] == "SENT"].copy()
    combined["description"] = combined["description"].fillna("")
    combined["raw_sms"] = combined["raw_sms"].fillna("")
    combined["text"] = (combined["description"] + " " + combined["raw_sms"]).str.strip()

    model = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ])
    model.fit(combined["text"], combined["category"])
    joblib.dump(model, output_path)
    return model, output_path


def retrain_prediction_models(
    user_history_dataset_path,
    output_dir=MODEL_DIR
):
    """Retrain XGBoost prediction models using accumulated user financial history.

    Expected target columns: target_month_end_expense, target_month_end_income.
    Non-feature columns are excluded automatically.
    """
    if XGBRegressor is None:
        raise ImportError("xgboost is required. Install with: pip install xgboost")

    data = pd.read_csv(user_history_dataset_path)
    feature_cols = [
        c for c in data.columns
        if c not in ["user_id", "month", "target_month_end_expense", "target_month_end_income", "target_overspend_risk", "is_synthetic_demo"]
    ]

    X = data[feature_cols]
    y_exp = data["target_month_end_expense"]
    y_inc = data["target_month_end_income"]

    exp_model = XGBRegressor(n_estimators=40, max_depth=3, learning_rate=0.08, random_state=42, objective="reg:squarederror")
    inc_model = XGBRegressor(n_estimators=40, max_depth=3, learning_rate=0.08, random_state=42, objective="reg:squarederror")

    exp_model.fit(X, y_exp)
    inc_model.fit(X, y_inc)

    exp_path = os.path.join(output_dir, "smartspend_expense_prediction_model_retrained.joblib")
    inc_path = os.path.join(output_dir, "smartspend_income_prediction_model_retrained.joblib")
    joblib.dump(exp_model, exp_path)
    joblib.dump(inc_model, inc_path)

    return {"expense_model_path": exp_path, "income_model_path": inc_path, "feature_columns": feature_cols}

print("Retraining functions are ready for backend integration.")

## 11. Example inference functions

In [ ]:
def predict_category(raw_sms, description=""):
    text = f"{description} {raw_sms}".strip()
    label = category_model.predict([text])[0]
    confidence = float(category_model.predict_proba([text]).max())
    return {"category": label, "confidence": confidence}


def predict_financial_outcome(feature_dict):
    row = pd.DataFrame([feature_dict])[feature_cols]
    predicted_expense = float(expense_model.predict(row)[0])
    predicted_income = float(income_model.predict(row)[0])
    overspend_risk = calculate_overspend_risk(predicted_expense, predicted_income)
    return {
        "predicted_month_end_expense": predicted_expense,
        "predicted_month_end_income": predicted_income,
        "overspend_risk": overspend_risk
    }

example_sms = expense_df.iloc[0]["raw_sms"]
example_description = expense_df.iloc[0]["description"]
print("Example categorisation:")
print(predict_category(example_sms, example_description))

example_features = X_test_p.iloc[0].to_dict()
print("Example financial prediction:")
print(predict_financial_outcome(example_features))